# Preprocesamiento y Preparación de Datos

**Caso de Estudio:** Ecommerce Customer Dataset
**Asignatura:** MLY1101 — Machine Learning
**Laboratorio de Modelamiento Supervisado: Regresión y Clasificación**

---

## Descripción

Este notebook corresponde a la **Fase 3 (Preparación de datos)** de la metodología CRISP-DM
aplicada al caso de estudio Ecommerce Customer Dataset. Toma como punto de partida los hallazgos
documentados en [`analisis_exploratorio.ipynb`](analisis_exploratorio.ipynb) (sin duplicados ni
nulos, outliers informativos en Age, relación no lineal de NumOfProducts y Age con el
abandono, Balance inflado en cero) y construye, mediante Pipeline/ColumnTransformer de
scikit-learn, el flujo de transformación que deja los datos listos para los dos problemas de
modelamiento:

- **Clasificación** — target: **Exited** (abandono del cliente).
- **Regresión** — target: **Balance** (saldo del cliente).

**Alcance:** este notebook cubre únicamente la preparación y transformación de datos. El
entrenamiento y la evaluación de los modelos están en [`modelamiento.ipynb`](modelamiento.ipynb).

Mantiene el uso de pipeline como buena práctica de la industria.

---

## Requisitos de Software

Este notebook fue desarrollado con Python 3.12. Bibliotecas necesarias:

- pandas (>=1.1.0)
- numpy (>=2.0.0)
- scikit-learn (>=1.3)

# Importación de librerías

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reconstrucción del dataset

El EDA confirmó que el dataset no tiene filas duplicadas, clientes repetidos, valores nulos ni
valores fuera de rango, por lo que no hay limpieza que reproducir: solo se descartan las columnas
que no son atributos del cliente. Se carga desde el CSV crudo para que este notebook sea
autocontenido y ejecutable de forma independiente.

- **RowNumber:** número de fila del archivo.
- **CustomerId y Surname:** identificadores del cliente. Surname tiene 2.932 valores distintos
  sobre 10.000 filas; no aporta valor predictivo generalizable y es un dato personal (ver la nota
  ética del resumen).

In [2]:
df = pd.read_csv('../data/ecommerce_customer_data.csv')
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

print(f'Filas: {len(df)} | Columnas: {df.shape[1]}')
df.head()

Filas: 10000 | Columnas: 11


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


# Fase 3 — Preparación de datos

El resto del notebook desarrolla la preparación en cinco pasos:

1. Ingeniería de características detectadas en el EDA.
2. Definición de variables (targets, features) y partición train/test.
3. Transformador personalizado de resguardo (multicolinealidad).
4. Ensamblado de los pipelines de preparación (uno por problema).
5. Ajuste sobre el conjunto de entrenamiento y verificación del resultado.

## 1. Ingeniería de características

Se crean dos variables derivadas, como respuesta a hallazgos de
[`analisis_exploratorio.ipynb`](analisis_exploratorio.ipynb):

- **distancia_edad_riesgo** = |Age − 55| (EDA, Sección 4.3): el abandono tiene forma de U
  invertida con la edad, con el máximo cerca de los 55 años. La distancia a 55 correlaciona con
  Exited r ≈ -0,36, contra r ≈ 0,29 de Age en bruto.
- **tiene_saldo** = 1 si Balance > 0 (EDA, Sección 5.4): los clientes con saldo abandonan el
  24,1% de las veces vs. 13,8% de los clientes con saldo 0. El EDA planteó revisar si esta
  variable agrega algo que Balance no tenga ya; se crea para comprobarlo en el paso 3. **Solo se
  usa en clasificación**: en regresión sería fuga de información, porque se calcula a partir del
  propio target Balance.

In [3]:
df['distancia_edad_riesgo'] = (df['Age'] - 55).abs()
df['tiene_saldo'] = (df['Balance'] > 0).astype(int)

print(f"r(Age, Exited)                   = {df['Age'].corr(df['Exited']):+.4f}")
print(f"r(distancia_edad_riesgo, Exited) = {df['distancia_edad_riesgo'].corr(df['Exited']):+.4f}")
print(f"r(Age, distancia_edad_riesgo)    = {df['Age'].corr(df['distancia_edad_riesgo']):+.4f}")
print(f"r(Balance, tiene_saldo)          = {df['Balance'].corr(df['tiene_saldo']):+.4f}")

df[['distancia_edad_riesgo', 'tiene_saldo']].describe()

r(Age, Exited)                   = +0.2853
r(distancia_edad_riesgo, Exited) = -0.3593
r(Age, distancia_edad_riesgo)    = -0.8309
r(Balance, tiene_saldo)          = +0.9228


,distancia_edad_riesgo,tiene_saldo
count,10000.000000,10000.000000
mean,17.435000,0.638300
std,8.032361,0.480517
min,0.000000,0.000000
25%,12.000000,0.000000
50%,18.000000,1.000000
75%,23.000000,1.000000
max,37.000000,1.000000


Age y distancia_edad_riesgo están correlacionadas entre sí (-0,83), algo esperable porque la
mayoría de los clientes tiene menos de 55 años. Se conservan ambas: la correlación no supera el
umbral de 0,9 del filtro de colinealidad definido más abajo, y cada una aporta información
distinta (Age la tendencia general, la distancia la caída del abandono en los mayores de 60).

tiene_saldo, en cambio, correlaciona 0,92 con Balance: como el 36% de los clientes tiene saldo
exactamente 0, Balance ya separa por sí solo a los clientes con y sin saldo, y la bandera casi
no agrega información nueva. Se deja que sea el filtro de colinealidad (paso 3) el que lo
detecte y decida, en vez de descartarla a mano.

## 2. Definición de variables y partición train/test

**Features numéricas** (se escalan): CreditScore, Age, Tenure, Balance,
EstimatedSalary, distancia_edad_riesgo.

**Features categóricas** (one-hot): Geography, Gender y NumOfProducts. Esta última es
numérica en el archivo, pero el EDA mostró que su relación con el abandono no es ordinal (1
producto: 27,7%, 2: 7,6%, 3: 82,7%, 4: 100%), por lo que se trata como categoría.

**Features binarias** (sin transformar, ya están en 0/1): HasCrCard, IsActiveMember,
tiene_saldo.

| Problema | Target | Features excluidas | Motivo |
|---|---|---|---|
| Clasificación | Exited | — | Usa todas las features. |
| Regresión | Balance | Balance, tiene_saldo, Exited | Las dos primeras son el propio target. Exited es un resultado posterior al saldo, no un atributo disponible al predecirlo. |

Se hace **una sola partición** 80/20, estratificada por Exited (20,4% de abandono), y se reutiliza
para los dos problemas. La partición se realiza antes de ajustar cualquier transformador, para
que las estadísticas de imputación y escalado se aprendan solo del conjunto de entrenamiento y
no se filtre información del conjunto de prueba (data leakage).

In [4]:
target_clasificacion = 'Exited'
target_regresion = 'Balance'

features_num = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'distancia_edad_riesgo']
features_cat = ['Geography', 'Gender', 'NumOfProducts']
features_bin = ['HasCrCard', 'IsActiveMember', 'tiene_saldo']

# Regresión: se quitan el target y lo que se deriva de él
features_num_reg = [col for col in features_num if col != target_regresion]
features_bin_reg = [col for col in features_bin if col != 'tiene_saldo']

X = df[features_num + features_cat + features_bin]
y_clf = df[target_clasificacion]
y_reg = df[target_regresion]

X_train, X_test, y_clf_train, y_clf_test, y_reg_train, y_reg_test = train_test_split(
    X, y_clf, y_reg, test_size=0.2, stratify=y_clf, random_state=42
)

print(f'Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}')
print(f'% abandono -> entrenamiento: {y_clf_train.mean()*100:.2f}% | prueba: {y_clf_test.mean()*100:.2f}%')

Entrenamiento: (8000, 12) | Prueba: (2000, 12)
% abandono -> entrenamiento: 20.38% | prueba: 20.35%


La estratificación mantiene la misma proporción de abandono (~20,4%) en ambos conjuntos.

## 3. Transformador personalizado

- **CorrelationFilter**: elimina columnas con correlación absoluta por sobre un umbral, como
  resguardo frente a multicolinealidad. La matriz de correlación del EDA no mostró pares de
  atributos originales por sobre 0,3, pero las variables derivadas sí se acercan al umbral de 0,9:
  distancia_edad_riesgo llega a 0,83 con Age (se conserva) y tiene_saldo a 0,92 con
  Balance (se espera que la descarte). Trabaja sobre DataFrames para conservar los nombres de
  las columnas.

**¿Por qué no se usa un Winsorizer (recorte de outliers)?** El EDA midió los outliers con la
regla IQR: solo Age (3,59%, hacia arriba) y CreditScore (0,15%, hacia abajo) los tienen, y
NumOfProducts = 4 (0,6%) es una categoría. Recortar Age sería contraproducente: los clientes
mayores abandonan mucho menos que los de 51 a 60 años (31,4% entre 61 y 70, y 8,3% sobre 70,
contra 56,2%), y un recorte los fusionaría con ese grupo de alto riesgo. Los de CreditScore son valores válidos de la escala y
demasiado pocos para afectar al escalado.

In [5]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad). Las columnas a eliminar se
    aprenden en fit (conjunto de entrenamiento) y se aplican igual en transform.
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        return pd.DataFrame(X).drop(columns=self.columns_to_drop_, errors='ignore')

## 4. Ensamblado de los pipelines de preparación

**Pipeline numérico:** SimpleImputer(strategy='median') → StandardScaler. El dataset no
tiene nulos, pero el imputador deja el pipeline seguro ante datos nuevos incompletos. Se usa la
mediana porque Balance es bimodal (36% en 0) y su media no representa a ningún cliente real.

**Pipeline categórico:** SimpleImputer(strategy='most_frequent') → OneHotEncoder.
drop='first' evita la trampa de las variables dummy y handle_unknown='ignore' deja el
pipeline seguro ante una categoría no vista (por ejemplo, un país nuevo).

**Binarias:** passthrough, ya están codificadas como 0/1.

Como los dos problemas usan columnas distintas, se define una función que arma el pipeline a
partir de las listas de columnas, en vez de repetir el código.

In [6]:
def construir_pipeline_preparacion(num_cols, cat_cols, bin_cols):
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ])

    preprocesador = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer,     num_cols),
            ('cat', categorical_transformer, cat_cols),
            ('bin', 'passthrough',           bin_cols)
        ],
        remainder='drop'
    ).set_output(transform='pandas')   # salida como DataFrame, con nombres de columna legibles

    return Pipeline(steps=[
        ('preprocesador', preprocesador),
        ('colinealidad',  CorrelationFilter(threshold=0.9))
    ])


pipeline_clasificacion = construir_pipeline_preparacion(features_num, features_cat, features_bin)
pipeline_regresion = construir_pipeline_preparacion(features_num_reg, features_cat, features_bin_reg)

pipeline_clasificacion

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'EstimatedSalary',
                                                   'distancia_edad_riesgo']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Geography', 'Gender',
                                                   'NumOfProducts']),
                                                 ('bin', 'passthrough',
                                                  ['HasCrCard',
                                                   'IsActiveMember',
                                                   'tiene_saldo'])])),
                ('colinealidad', CorrelationFilter())])

## 5. Ajuste sobre el conjunto de entrenamiento

In [7]:
X_clf_train_prep = pipeline_clasificacion.fit_transform(X_train, y_clf_train)
X_clf_test_prep = pipeline_clasificacion.transform(X_test)

X_reg_train_prep = pipeline_regresion.fit_transform(X_train, y_reg_train)
X_reg_test_prep = pipeline_regresion.transform(X_test)

for nombre, pipe, tr, te in [
    ('Clasificación', pipeline_clasificacion, X_clf_train_prep, X_clf_test_prep),
    ('Regresión', pipeline_regresion, X_reg_train_prep, X_reg_test_prep),
]:
    print(f'{nombre}:')
    print(f"  Columnas descartadas por colinealidad (umbral 0.9): {pipe.named_steps['colinealidad'].columns_to_drop_}")
    print(f'  Forma final -> Entrenamiento: {tr.shape} | Prueba: {te.shape}')
    print(f'  Columnas: {list(tr.columns)}\n')

Clasificación:
  Columnas descartadas por colinealidad (umbral 0.9): ['bin__tiene_saldo']
  Forma final -> Entrenamiento: (8000, 14) | Prueba: (2000, 14)
  Columnas: ['num__CreditScore', 'num__Age', 'num__Tenure', 'num__Balance', 'num__EstimatedSalary', 'num__distancia_edad_riesgo', 'cat__Geography_Germany', 'cat__Geography_Spain', 'cat__Gender_Male', 'cat__NumOfProducts_2', 'cat__NumOfProducts_3', 'cat__NumOfProducts_4', 'bin__HasCrCard', 'bin__IsActiveMember']

Regresión:
  Columnas descartadas por colinealidad (umbral 0.9): []
  Forma final -> Entrenamiento: (8000, 13) | Prueba: (2000, 13)
  Columnas: ['num__CreditScore', 'num__Age', 'num__Tenure', 'num__EstimatedSalary', 'num__distancia_edad_riesgo', 'cat__Geography_Germany', 'cat__Geography_Spain', 'cat__Gender_Male', 'cat__NumOfProducts_2', 'cat__NumOfProducts_3', 'cat__NumOfProducts_4', 'bin__HasCrCard', 'bin__IsActiveMember']



In [8]:
print(f'Valores nulos en X_clf_train_prep: {X_clf_train_prep.isna().sum().sum()}')
print(f'Valores nulos en X_clf_test_prep: {X_clf_test_prep.isna().sum().sum()}')
print(f'Valores nulos en X_reg_train_prep: {X_reg_train_prep.isna().sum().sum()}')
print(f'Valores nulos en X_reg_test_prep: {X_reg_test_prep.isna().sum().sum()}')

X_clf_train_prep.describe().round(3)

Valores nulos en X_clf_train_prep: 0
Valores nulos en X_clf_test_prep: 0
Valores nulos en X_reg_train_prep: 0
Valores nulos en X_reg_test_prep: 0


,num__CreditScore,num__Age,num__Tenure,num__Balance,num__EstimatedSalary,num__distancia_edad_riesgo,cat__Geography_Germany,cat__Geography_Spain,cat__Gender_Male,cat__NumOfProducts_2,cat__NumOfProducts_3,cat__NumOfProducts_4,bin__HasCrCard,bin__IsActiveMember
count,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000
mean,-0.000,0.000,-0.000,0.000,-0.000,0.000,0.251,0.250,0.548,0.459,0.026,0.006,0.709,0.515
std,1.000,1.000,1.000,1.000,1.000,1.000,0.433,0.433,0.498,0.498,0.159,0.079,0.454,0.500
min,-3.114,-1.990,-1.732,-1.226,-1.739,-2.177,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,-0.691,-0.660,-0.696,-1.226,-0.855,-0.680,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.013,-0.185,-0.006,0.332,-0.005,0.069,0.000,0.000,1.000,0.000,0.000,0.000,1.000,1.000
75%,0.686,0.480,1.030,0.823,0.851,0.693,1.000,1.000,1.000,1.000,0.000,0.000,1.000,1.000
max,2.063,5.040,1.720,2.600,1.749,2.441,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


Las columnas numéricas quedan con media 0 y desviación 1 en el conjunto de entrenamiento y las
categóricas como indicadores 0/1.

Como se anticipó, el filtro de colinealidad descartó tiene_saldo en el pipeline de
clasificación (correlación 0,92 con Balance): la señal "tener saldo aumenta el abandono" que
se vio en el EDA ya está contenida en Balance. En regresión no se descarta nada.

# Resumen

- Se reconstruyó el dataset desde el CSV crudo de forma autocontenida, descartando RowNumber,
  CustomerId y Surname. No hubo que eliminar ni consolidar filas: el EDA confirmó que no
  hay duplicados, nulos ni valores fuera de rango.
- Se agregaron dos variables de ingeniería de características: distancia_edad_riesgo
  (|Age − 55|, captura la forma de U invertida del abandono con la edad), que se conserva, y
  tiene_saldo, que el filtro de colinealidad descartó por ser redundante con Balance
  (r = 0,92).
- NumOfProducts se trata como categórica por su relación no ordinal con el abandono.
- No se recortan outliers: los de Age son reales e informativos.
- Se hizo una partición 80/20 estratificada por Exited, reutilizada en los dos problemas, y
  se construyeron dos pipelines de preparación (imputación, escalado, one-hot y filtro de
  colinealidad) ajustados exclusivamente sobre el conjunto de entrenamiento.
- Resultado: **8.000 clientes de entrenamiento / 2.000 de prueba**, 14 columnas para
  clasificación y 13 para regresión, sin valores nulos.
- **Nota ética/privacidad:** Surname y CustomerId identifican a personas reales y se
  excluyen de las features. Gender y Geography sí se usan como predictores; sus
  implicancias (posible trato diferenciado por género o país) deben evaluarse antes de
  usar los modelos en decisiones reales.

El resultado queda listo para la etapa de modelamiento, en
[`modelamiento.ipynb`](modelamiento.ipynb).